In [2]:
import os
import sys

ROOT_PATH = '../'
os.chdir(ROOT_PATH)

sys.path.append(os.path.abspath(ROOT_PATH))

lib_path = os.path.abspath(os.path.join(ROOT_PATH, 'lib'))
if lib_path not in sys.path:
  sys.path.append(lib_path)

In [3]:
import torch
import numpy as np
from ultralytics import YOLO

In [4]:
model = YOLO('model/weights/yolo26m-sem.pt').to('cuda')

In [5]:
dummy = torch.from_numpy(np.zeros((1, 3, 640, 640))).to(dtype=torch.float, device='cuda')
dummy.size()

torch.Size([1, 3, 640, 640])

In [6]:
candidates = [2, 4, 6, 10]

for idx in candidates:
  model.model.model[idx].register_forward_hook(lambda module, input, output: print(output.size()))

In [7]:
test = model.model(dummy)

torch.Size([1, 256, 160, 160])
torch.Size([1, 512, 80, 80])
torch.Size([1, 512, 40, 40])
torch.Size([1, 512, 20, 20])


In [5]:
list(model.named_modules())

[('',
  YOLO(
    (model): SemanticSegmentationModel(
      (model): Sequential(
        (0): Conv(
          (conv): Conv2d(3, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
          (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
          (act): SiLU(inplace=True)
        )
        (1): Conv(
          (conv): Conv2d(64, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
          (bn): BatchNorm2d(128, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
          (act): SiLU(inplace=True)
        )
        (2): C3k2(
          (cv1): Conv(
            (conv): Conv2d(128, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
            (bn): BatchNorm2d(128, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
            (act): SiLU(inplace=True)
          )
          (cv2): Conv(
            (conv): Conv2d(192, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
            (bn): B

In [9]:
depth_array = np.load('data/RescueNet/train/train-depth-img/10780.npy')

In [13]:
depth_array.min(), depth_array.max()

(np.float32(0.9368103), np.float32(1.2412717))

In [14]:
print(depth_array.shape)

(640, 640)


In [22]:
def minmax_normalize(matrix: np.ndarray) -> np.array:
  d_min, d_max = matrix.min(), matrix.max()
  
  return (matrix - d_min) / (d_max - d_min)

normalized = minmax_normalize(depth_array)
normalized.min(), normalized.max()

(np.float32(0.0), np.float32(1.0))